# Case Overview
This project analyzes the Instacart Grocery Orders dataset (~3.4M orders).
The goal is to understand customer behavior, product demand, and reorder loyalty.

# Problem Statement
We want to answer:
1. Which departments sell the most?
2. Which products are reordered most?
3. What are the busiest days and hours?
4. What is the average basket size?
5. Do customers order regularly or randomly?


In [2]:
import pandas as pd

# قراءة كل ملفات CSV من فولدر data
orders = pd.read_csv("../data/orders.csv")
order_products_prior = pd.read_csv("../data/order_products__prior.csv")
order_products_train = pd.read_csv("../data/order_products__train.csv")
products = pd.read_csv("../data/products.csv")
aisles = pd.read_csv("../data/aisles.csv")
departments = pd.read_csv("../data/departments.csv")

# نتأكد من حجم البيانات
print("Orders:", orders.shape)
print("Prior:", order_products_prior.shape)
print("Train:", order_products_train.shape)
print("Products:", products.shape)
print("Aisles:", aisles.shape)
print("Departments:", departments.shape)


Orders: (3421083, 7)
Prior: (32434489, 4)
Train: (1384617, 4)
Products: (49688, 4)
Aisles: (134, 2)
Departments: (21, 2)


In [3]:
# عدد الصفوف باستخدام Pandas
print("Pandas count:", orders.shape[0])

# عدد الصفوف باستخدام قراءة مباشرة للملف
with open("../data/orders.csv") as f:
    lines = f.readlines()
print("Raw file count:", len(lines))


Pandas count: 3421083
Raw file count: 3421084


# Question 1: Which departments sell the most?


In [4]:
# دمج المنتجات مع الأقسام
products_full = products.merge(aisles, on="aisle_id").merge(departments, on="department_id")

# دمج الطلبات كلها (prior + train)
order_products_all = pd.concat([order_products_prior, order_products_train])

# ربط الطلبات بالمنتجات
merged = order_products_all.merge(products_full, on="product_id")

# حساب عدد المنتجات في كل قسم
top_departments = merged["department"].value_counts().head(10)
top_departments


department
produce            9888378
dairy eggs         5631067
snacks             3006412
beverages          2804175
frozen             2336858
pantry             1956819
bakery             1225181
canned goods       1114857
deli               1095540
dry goods pasta     905340
Name: count, dtype: int64

### Interpretation
The top-selling department is **Produce**, followed by **Dairy Eggs** and **Snacks**.
This shows that most orders focus on daily essentials like fruits, vegetables, and dairy products.


## Question 2: Which products are reordered most?

In [5]:
# دمج الطلبات مع المنتجات
merged = order_products_all.merge(products_full, on="product_id")

# حساب عدد مرات إعادة الطلب لكل منتج
reordered_products = merged[merged["reordered"] == 1]["product_name"].value_counts().head(10)
reordered_products


product_name
Banana                    415166
Bag of Organic Bananas    329275
Organic Strawberries      214448
Organic Baby Spinach      194939
Organic Hass Avocado      176173
Organic Avocado           140270
Organic Whole Milk        118684
Large Lemon               112178
Organic Raspberries       109688
Strawberries              104588
Name: count, dtype: int64

### Interpretation
The most reordered products are daily essentials such as bananas, milk, and eggs.
This shows strong customer loyalty to staple items that are purchased repeatedly.


## Question 3: What are the busiest days and hours?


In [6]:
# حساب عدد الطلبات حسب اليوم
orders_by_day = orders["order_dow"].value_counts().sort_index()

# حساب عدد الطلبات حسب الساعة
orders_by_hour = orders["order_hour_of_day"].value_counts().sort_index()

print("Orders by Day:\n", orders_by_day)
print("\nOrders by Hour:\n", orders_by_hour)


Orders by Day:
 order_dow
0    600905
1    587478
2    467260
3    436972
4    426339
5    453368
6    448761
Name: count, dtype: int64

Orders by Hour:
 order_hour_of_day
0      22758
1      12398
2       7539
3       5474
4       5527
5       9569
6      30529
7      91868
8     178201
9     257812
10    288418
11    284728
12    272841
13    277999
14    283042
15    283639
16    272553
17    228795
18    182912
19    140569
20    104292
21     78109
22     61468
23     40043
Name: count, dtype: int64


### Interpretation
Most orders happen on **day 0 (Sunday)** and **day 1 (Monday)**.
Peak hours are between **10 AM and 3 PM**, showing customers shop mostly during daytime.


## Question 4: What is the average basket size?


In [7]:
# حساب عدد المنتجات في كل طلب
basket_size = order_products_all.groupby("order_id")["product_id"].count()

# المتوسط
average_basket = basket_size.mean()
average_basket


np.float64(10.10707325550502)

### Interpretation
The average basket size is around **10–12 products per order**.
This shows customers usually buy multiple items in one shopping trip.


## Question 5: Do customers order regularly or randomly?


In [8]:
# توزيع الأيام منذ آخر طلب
days_distribution = orders["days_since_prior_order"].value_counts().sort_index()
days_distribution


days_since_prior_order
0.0      67755
1.0     145247
2.0     193206
3.0     217005
4.0     221696
5.0     214503
6.0     240013
7.0     320608
8.0     181717
9.0     118188
10.0     95186
11.0     80970
12.0     76146
13.0     83214
14.0    100230
15.0     66579
16.0     46941
17.0     39245
18.0     35881
19.0     34384
20.0     38527
21.0     45470
22.0     32012
23.0     23885
24.0     20712
25.0     19234
26.0     19016
27.0     22013
28.0     26777
29.0     19191
30.0    369323
Name: count, dtype: int64

### Interpretation
Most customers reorder after **7 days**, showing weekly shopping habits.
Some variation exists, but the majority follow a regular weekly pattern.


# Future Work
We can build a Reorder Prediction Model using features like:
- order_number
- days_since_prior_order
- add_to_cart_order
- department / aisle
- reordered

This model can help recommend products to customers based on their history.


In [15]:
# ناخد عينة من البيانات علشان الرامات
sample_data = model_data.sample(n=200000, random_state=42)

X = sample_data.drop(columns=["reordered"])
y = sample_data["reordered"]


In [16]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [17]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

# بناء الموديل
model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
model.fit(X_train, y_train)

# التنبؤ
y_pred = model.predict(X_test)

# تقييم الموديل
print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))


Accuracy: 0.709775
F1 Score: 0.7882458092406471


## Future Work: Reorder Prediction Model

We built a simple machine learning model (Random Forest) to predict whether a product will be reordered.  
Features used:
- order_number
- days_since_prior_order
- add_to_cart_order
- department / aisle (encoded)

### Results
- Accuracy: ~71%
- F1 Score: ~79%

### Interpretation
The model shows promising performance, indicating that customer reorder behavior can be predicted with reasonable accuracy.  
This can be extended to build personalized product recommendations for customers.
